# FORGE M6 — AMP benchmark: FP32 vs FP16 on one T4

**Purpose:** measure whether FP16 mixed precision (AMP) materially improves
real T4 training throughput, and whether it stays numerically stable. This
is the experiment EXP-005 pointed at: FP32 training projects to ~218
GPU-hours for 125M/2.5B tokens, which is not viable on free Kaggle quota.

**Fair-comparison rules (enforced by the cells below, not by goodwill):**
- Identical model (25.3M), dataset, seq len, batch, optimizer, LR schedule,
  and measured steps for FP32 and FP16. The ONLY difference is `--precision`.
- Warm-up steps (20) are excluded from throughput via `--warmup-steps`;
  the reported number is `steady_tok_per_s`, measured over a CUDA-synchronized
  region. Overall `tok_per_s` is reported only for reference.
- Single-GPU runs only. DDP benchmarking is a SEPARATE follow-up after
  single-GPU AMP is stable — do not compare these numbers with Run C from
  EXP-005 (different step counts, different global batch).
- FP32 and FP16 evidence live in separate directories
  (`checkpoints/amp_bench/fp32` vs `checkpoints/amp_bench/fp16`) and every
  summary carries its `precision` label.

**What this does NOT prove:** matching toy-data loss is a stability check,
not coding capability. A 25M→125M projection is an estimate with a scaling
caveat, not a measurement. FP16 does NOT necessarily yield 8x — the measured
ratio is what counts.

**Expected GPU budget:** FP32 300 steps + FP16 300 steps + resume test,
each a few minutes — well under 30 minutes total.

In [ ]:
import torch, subprocess, json, os, sys
print("python:", sys.version.split()[0])
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("cuda version (torch):", torch.version.cuda if torch.cuda.is_available() else None)
print("cudnn:", torch.backends.cudnn.version() if torch.cuda.is_available() else None)
n = torch.cuda.device_count()
print("visible GPUs:", n)
for i in range(n):
    props = torch.cuda.get_device_properties(i)
    free, total = torch.cuda.mem_get_info(i)
    print(f"  gpu{i}: {props.name} | total VRAM {total/1e9:.1f} GB | free {free/1e9:.1f} GB")
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"],
      capture_output=True, text=True).stdout)
assert torch.cuda.is_available(), "NO CUDA — enable the GPU accelerator in notebook settings first"
print("ENV CHECK: CUDA is usable")
try:
    sc = torch.amp.GradScaler("cuda")
    print("GradScaler available, initial scale:", sc.get_scale())
except Exception as e:
    raise AssertionError(f"GradScaler unavailable: {e}")

## Get the code

Same pattern as EXP-005: clone from GitHub (already pushed, `REPO_URL` is
pre-filled) and pin to an exact commit. The pin is filled in before this
notebook is handed over — never a moving branch.

In [ ]:
import os, shutil, subprocess, zipfile

def sh(cmd):
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError('command failed: ' + ' '.join(cmd) + '\n' + r.stderr[-2000:])
    return r.stdout.strip()

# --- Set these once, then Run All. Values are strings, so they need quotes. ---
REPO_URL = "https://github.com/mohnish-ydv/forge"  # Option A: clone from GitHub
FORGE_COMMIT = "7d3e58ff4eb48853e2107d74ab4a8ad2d8874f09"  # pinned GitHub commit — never a moving branch
ZIP_PATH = ""  # Option B only: path to an uploaded forge zip, else ""

REPO_DIR = '/kaggle/working/forge'
if REPO_URL:
    if not os.path.isdir(REPO_DIR):
        sh(['git', 'clone', REPO_URL, REPO_DIR])
        print('cloned', REPO_URL)
    else:
        print('repo dir already exists — reusing it')
elif ZIP_PATH:
    zipfile.ZipFile(ZIP_PATH).extractall(REPO_DIR)
    if not os.path.isdir(os.path.join(REPO_DIR, 'src')):
        subs = [d for d in os.listdir(REPO_DIR) if os.path.isdir(os.path.join(REPO_DIR, d))]
        if len(subs) == 1 and os.path.isdir(os.path.join(REPO_DIR, subs[0], 'src')):
            for item in os.listdir(os.path.join(REPO_DIR, subs[0])):
                shutil.move(os.path.join(REPO_DIR, subs[0], item), os.path.join(REPO_DIR, item))
            os.rmdir(os.path.join(REPO_DIR, subs[0]))
    print('unzipped', ZIP_PATH)
else:
    raise SystemExit('Set REPO_URL (Option A) or ZIP_PATH (Option B) above, then re-run')
print('code dir:', REPO_DIR)

In [ ]:
os.chdir('/kaggle/working/forge')
actual = None
if REPO_URL:
    # (re-)enforce the pin even if a stale checkout already exists
    sh(['git', 'fetch', 'origin', FORGE_COMMIT])
    sh(['git', 'checkout', '--force', FORGE_COMMIT])
    sh(['git', 'clean', '-fdx', '-e', 'checkpoints'])  # drop stale untracked files, keep ckpts
    actual = sh(['git', 'rev-parse', 'HEAD'])
    assert actual.startswith(FORGE_COMMIT), f'pinned {FORGE_COMMIT} but HEAD is {actual}'
    print('pinned commit verified:', actual)
else:
    print('ZIP path: commit pinning not verifiable — record your repo state manually')
print('ACTUAL_HEAD=' + str(actual))

## Toy data (deterministic, sha-pinned)

Same generator as M3/EXP-005 (`src/toy_data.py`), 65,536 sequences
(~2.5M tokens). This is a throughput + stability fixture, not a corpus.

In [ ]:
import sys; sys.path.insert(0, 'src')
from toy_data import write_toy_dataset, load_ids
info = write_toy_dataset('data/toy/train.bin', n_sequences=65536)
ids = load_ids('data/toy/train.bin')
print('sha256:', info['sha256'])
print('tokens:', len(ids))
# n_sequences=65536 sha (M3's 1ea08995... is the 256-sequence default — different size, different file)
assert info['sha256'] == 'cab143199e42e84fadb51c75b5475dea370258a00bc3030da516c204bd78a8e7', 'toy data changed!'

## Benchmark design — identical config, only precision differs

| setting | FP32 run | FP16 run |
|---|---|---|
| model | 25.3M (d=512, L=8, 8 heads / 4 KV) | same |
| batch × seq | 16 × 512 = 8,192 tokens/step | same |
| steps | 300 | 300 |
| warmup (excluded) | 20 steps | 20 steps |
| optimizer / LR | AdamW, lr 3e-4, wd 0.1, cosine T_max=300 | same |
| val | at step 300 only (one val pass inside the measured region, identical for both) | same |
| checkpoint dir | `checkpoints/amp_bench/fp32` | `checkpoints/amp_bench/fp16` |
| precision flag | `--precision fp32` | `--precision fp16` |

The headline number is `steady_tok_per_s` from each run's `last_summary.json`
(post-warmup, CUDA-synchronized). `tok_per_s` (whole run incl. warmup) is
reported for reference only.

In [ ]:
import subprocess, json, os

def run_bench(precision, ckpt_dir, steps=300):
    cmd = ['python', 'src/train.py', '--precision', precision,
           '--steps', str(steps), '--t-max', str(steps), '--warmup-steps', '20',
           '--device', 'cuda', '--d-model', '512', '--n-layers', '8',
           '--n-heads', '8', '--n-kv-heads', '4',
           '--batch-size', '16', '--seq-len', '512',
           '--ckpt-dir', ckpt_dir, '--log-every', '50',
           '--ckpt-every', str(steps), '--val-every', str(steps)]
    r = subprocess.run(cmd, capture_output=True, text=True)
    print(r.stdout[-1200:])
    assert 'device=cuda' in r.stdout, 'SILENT CPU FALLBACK? device=cuda not in log'
    assert r.returncode == 0, f'{precision} run failed:\n' + r.stderr[-2000:]
    s = json.load(open(os.path.join(ckpt_dir, 'last_summary.json')))
    assert s['precision'] == precision, f"summary precision {s['precision']} != {precision}"
    print(f"{precision} OK — steady {s['steady_tok_per_s']} tok/s, "
          f"train loss {s['final_train_loss']:.4f}, val loss {s['final_val_loss']:.4f}")
    return s

s_fp32 = run_bench('fp32', 'checkpoints/amp_bench/fp32')

In [ ]:
s_fp16 = run_bench('fp16', 'checkpoints/amp_bench/fp16')
assert s_fp16['scaler_final_scale'] is not None, 'FP16 run has no scaler scale'
print('FP16 scaler final scale:', s_fp16['scaler_final_scale'])
print('FP16 skipped optimizer steps:', s_fp16['skipped_steps'],
      '| non-finite losses:', s_fp16['nonfinite_losses'],
      '| non-finite grads:', s_fp16['nonfinite_grads'])
print('FP16 peak allocated MB:', s_fp16['peak_gpu_mem_mb'],
      '| peak reserved MB:', s_fp16['peak_gpu_mem_reserved_mb'])

## AMP checkpoint-resume test (fresh processes)

Phase 1 trains FP16 for 20 steps (T_max=40), phase 2 resumes in a fresh
process to 40 steps and must restore optimizer, scheduler, scaler, RNG and
step count — proven by the `resumed scaler scale=` log line and a valid
40-step summary. A cross-precision resume (fp32 on the fp16 checkpoint)
must then be REFUSED loudly, not silently accepted.

In [ ]:
import math, subprocess, json, os

D = 'checkpoints/amp_bench/fp16_resume'
base = ['python', 'src/train.py', '--device', 'cuda',
        '--d-model', '512', '--n-layers', '8', '--n-heads', '8', '--n-kv-heads', '4',
        '--batch-size', '16', '--seq-len', '512', '--ckpt-dir', D,
        '--log-every', '10', '--val-every', '100']
r1 = subprocess.run(base + ['--precision', 'fp16', '--steps', '20', '--t-max', '40',
                            '--ckpt-every', '20'], capture_output=True, text=True)
print(r1.stdout[-800:])
assert r1.returncode == 0, 'resume phase 1 failed:\n' + r1.stderr[-2000:]
r2 = subprocess.run(base + ['--precision', 'fp16', '--resume', '--steps', '40', '--t-max', '40',
                            '--ckpt-every', '40'], capture_output=True, text=True)
print(r2.stdout[-800:])
assert r2.returncode == 0, 'resume phase 2 failed:\n' + r2.stderr[-2000:]
assert 'resumed scaler scale=' in r2.stdout, 'scaler restore not logged — resume did not restore scaler'
s_res = json.load(open(os.path.join(D, 'last_summary.json')))
assert s_res['steps'] == 40 and s_res['precision'] == 'fp16' and s_res['scaler_final_scale'] is not None
assert math.isfinite(s_res['final_train_loss']) and math.isfinite(s_res['final_val_loss'])
print('RESUME OK — 40 steps, scaler restored, finite losses')
# cross-precision resume must be refused
r3 = subprocess.run(base + ['--precision', 'fp32', '--resume', '--steps', '40', '--t-max', '40'],
                    capture_output=True, text=True)
assert r3.returncode != 0 and 'Precision mismatch' in r3.stderr, 'mismatch resume was NOT rejected!'
print('MISMATCH REJECTION OK — fp32 resume on fp16 checkpoint refused')

## Results → JSON, MFU, and 125M projection

**Methodology (explicit):**
- *Measured:* `steady_tok_per_s`, elapsed, peak allocated + reserved memory,
  losses, skipped steps, non-finite counts, scaler scale — from each run's
  `last_summary.json`.
- *Estimated:* FLOPs/token ≈ 6N + 12·L·d·S (N exact from the config, ±10%
  methodology band). MFU = measured steady tok/s × FLOPs/token ÷ peak.
- *Hardware peaks:* FP32 runs compare against the T4's **FP32 8.1 TFLOPS**;
  FP16 runs compare against the T4's **FP16 tensor-core 65 TFLOPS** — the
  precision actually used for matmuls. These are different denominators by
  design; comparing an FP16 MFU against the FP32 peak (or vice versa) would
  be a category error.
- *Scope:* single-GPU runs, so per-GPU = aggregate. DDP numbers are NOT in
  this file.
- *125M projection:* scaled from measured AMP steady tok/s by the
  FLOPs/token ratio (125M config at S=512, same assumption as EXP-005).
  Scaling 25M→125M is approximate (attention fraction, memory behavior and
  long-run effects differ) — treat it as an estimate with a band, and the
  budget below adds explicit overhead + failure contingency on top.

In [ ]:
import json, os, sys
sys.path.insert(0, 'src')
from model import ForgeConfig

gpu_names = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
vram = [torch.cuda.get_device_properties(i).total_memory for i in range(torch.cuda.device_count())]

def load(p):
    return json.load(open(p))

s32 = load('checkpoints/amp_bench/fp32/last_summary.json')
s16 = load('checkpoints/amp_bench/fp16/last_summary.json')
sres = load('checkpoints/amp_bench/fp16_resume/last_summary.json')

N25 = ForgeConfig(vocab_size=256, d_model=512, n_layers=8, n_heads=8, n_kv_heads=4, max_seq_len=513).param_count()['total']
N125 = ForgeConfig(vocab_size=256, d_model=768, n_layers=18, n_heads=12, n_kv_heads=3, max_seq_len=513).param_count()['total']
S = 512
fpt25 = 6 * N25 + 12 * 8 * 512 * S
fpt125 = 6 * N125 + 12 * 18 * 768 * S
assert N25 == 25305600, f'25M param count changed: {N25}'

speedup = s16['steady_tok_per_s'] / s32['steady_tok_per_s']
mfu32 = s32['steady_tok_per_s'] * fpt25 / 8.1e12
mfu16 = s16['steady_tok_per_s'] * fpt25 / 65e12
proj_hours = 2.5e9 * fpt125 / (s16['steady_tok_per_s'] * fpt25) / 3600

out = {
    "notebook": "forge_m6_amp_bench",
    "pinned_commit": FORGE_COMMIT,
    "actual_head": globals().get('actual'),
    "hardware": {
        "gpu_names": gpu_names,
        "gpu_count": len(gpu_names),
        "vram_total_mb": [round(v / 1e6, 1) for v in vram],
        "torch": torch.__version__,
        "cuda_torch": torch.version.cuda,
        "python": sys.version.split()[0],
        "fp32_peak_flops_used": 8.1e12,
        "fp16_peak_flops_used": 65e12,
    },
    "methodology": "measured: steady_tok_per_s (post-warmup, CUDA-synchronized), elapsed, peak allocated+reserved mem, losses, skipped steps, non-finite counts, scaler scale. estimated: FLOPs/token = 6N + 12*L*d*S (+-10% band). MFU vs the peak of the precision actually used. scope: per-GPU = aggregate (single-GPU runs).",
    "config_common": {
        "d_model": 512, "n_layers": 8, "n_heads": 8, "n_kv_heads": 4,
        "batch": 16, "seq": 512, "steps": 300, "warmup_steps": 20,
        "lr": 3e-4, "weight_decay": 0.1, "schedule": "cosine T_max=300",
        "optimizer": "AdamW", "params_exact": N25,
    },
    "runs": {"fp32": s32, "fp16": s16},
    "resume_test": {
        "status": "ok",
        "steps": sres['steps'],
        "precision": sres['precision'],
        "scaler_restored": sres['scaler_final_scale'] is not None,
        "scaler_final_scale": sres['scaler_final_scale'],
        "mismatch_rejected": True,
    },
    "comparison": {
        "steady_tok_per_s_fp32": s32['steady_tok_per_s'],
        "steady_tok_per_s_fp16": s16['steady_tok_per_s'],
        "steady_speedup_fp16_vs_fp32": round(speedup, 3),
        "mfu_fp32_vs_fp32_peak": round(mfu32, 4),
        "mfu_fp16_vs_fp16_peak": round(mfu16, 4),
        "scope": "per-GPU (single-GPU runs; aggregate == per-GPU)",
        "fp16_skipped_steps": s16['skipped_steps'],
        "fp16_nonfinite_losses": s16['nonfinite_losses'],
        "fp16_nonfinite_grads": s16['nonfinite_grads'],
        "note": "speedup is MEASURED (steady tok/s ratio). It is not 8x by assumption: 65/8.1 is a peak ratio for pure matmul, not a training speedup.",
    },
    "projection_125m_2p5B": {
        "hours_base_from_measured_amp": round(proj_hours, 1),
        "hours_with_overheads_15pct": round(proj_hours * 1.15, 1),
        "hours_planning_50pct": round(proj_hours * 1.5, 1),
        "assumptions": "125M config (d=768, L=18) at S=512, same FLOPs/token methodology as EXP-005; setup/validation/checkpointing ~15%; planning figure adds a partial failed/repeated run (~35% of base).",
        "caveat": "25M->125M scaling is approximate: attention fraction, memory behavior, and long-run effects differ. Treat as an estimate with a band, not a measurement.",
    },
    "status": "ok",
}
with open('/kaggle/working/amp_bench_results.json', 'w') as f:
    json.dump(out, f, indent=2, sort_keys=True)
print('=== AMP BENCHMARK VERDICT ===')
print(f"FP32 steady: {s32['steady_tok_per_s']} tok/s | MFU {mfu32:.3f} (vs 8.1 TFLOPS FP32)")
print(f"FP16 steady: {s16['steady_tok_per_s']} tok/s | MFU {mfu16:.3f} (vs 65 TFLOPS FP16)")
print(f"MEASURED speedup: {speedup:.2f}x (not assumed 8x)")
print(f"FP16 skipped steps: {s16['skipped_steps']} | non-finite losses: {s16['nonfinite_losses']} | grads: {s16['nonfinite_grads']}")
print(f"125M/2.5B projection from measured AMP: {proj_hours:.1f} h base, {proj_hours*1.5:.1f} h planning")
print('wrote /kaggle/working/amp_bench_results.json')

## After the run

1. **Download** `amp_bench_results.json` from the notebook output (right
   panel → `/kaggle/working`).
2. Paste it back to Khushi — the EXP-006 decision (AMP passes / conditional
   / fails) is made from measured numbers, not from this notebook's hopes.
3. **DDP + AMP benchmarking is deliberately NOT in this notebook.** It runs
   separately, after the single-GPU AMP verdict, with equal step counts for
   single vs multi-GPU (EXP-005's Run C comparison was confounded and will
   not be repeated).

**Troubleshooting:**
- `CUDA out of memory` → the FP16 run should use *less* memory than FP32;
  if it OOMs, something is wrong — report it, don't just shrink the batch.
- `Precision mismatch` on resume → you resumed a directory with the wrong
  `--precision`. Use a fresh `--ckpt-dir` per precision (the cells do).
- Many skipped steps / collapsing scale → AMP is unstable for this setup;
  that is a FAIL signal for the AMP verdict, not something to tune away
  silently.